In [1]:
import pandas as pd
import altair as alt
from pyarrow import lib
import numpy as np

In [2]:
## Reading Gameweek Data

dfs = []

for gw in range (1, 39):
    df = pd.read_csv(f"data/2025-2026/gameweeks/gw{gw}.csv")
    df["gameweek"] = gw
    dfs.append(df)

gameweek_data = pd.concat(dfs, ignore_index=True)

## gameweek_data.head()

In [24]:
## Reading in Other Datasets

player_ids = pd.read_csv("data/2025-2026/player_idlist.csv")
fixtures = pd.read_csv("data/2025-2026/fixtures.csv")
teams = pd.read_csv("data/2025-2026/teams.csv")
player_data = pd.read_csv("data/2025-2026/cleaned_players.csv")

## player_ids.head()
## fixtures.head()
## teams.head()
## player_data.head()



In [25]:
## Cleaning team data

teams = teams[['id', 'short_name', 'strength']]

teams = teams.rename(columns={'id': 'team_id', 'strength': 'team_strength'})



In [9]:
## Joining player data with player_id

player_data['name'] = player_data['first_name'] + ' ' + player_data['second_name']

player_data.insert(0, 'name', player_data.pop('name'))

player_data = player_data.drop(columns=['first_name', 'second_name'])


player_ids['name'] = player_ids['first_name'] + ' ' + player_ids['second_name']

player_ids = player_ids.drop(columns = ['first_name', 'second_name'])


player_data = pd.merge(player_ids, player_data, how = 'left', on = 'name')



In [10]:
gameweek_data = pd.merge(player_ids, gameweek_data, how = 'left', on = 'name')

In [ ]:
## Adding opponents

gameweek_data = gameweek_data.merge(teams, left_on='opponent_team', right_on='team_id').rename(
    columns={'short_name': 'opponent', 'team_strength': 'opponent_strength'}
    )

In [11]:
MIN_MINUTES = 450

player_options = sorted(
    player_data.loc[player_data['minutes'] > MIN_MINUTES, 'name'].unique()
)

player_select = alt.param(
    name='player_select',
    value=player_options[0],
    bind=alt.binding_select(options=player_options, name='Player: ')
)

In [ ]:
## Points by Gameweek

alt.data_transformers.enable('vegafusion')

week_by_week_data = gameweek_data

## teams = sorted(gameweek_data['team'].unique())
## players = sorted(gameweek_data['name'].unique())

week_by_week_data['team_player'] = week_by_week_data['team'] + '_' + week_by_week_data['name']
options = sorted(week_by_week_data['team_player'].unique())

selection = alt.param(name = 'team_player', value = options[0],
                        bind = alt.binding_select(options=options, name = "Team-Player: "))

points_by_gameweek = (
    alt.Chart(week_by_week_data)
    .mark_line(point=alt.OverlayMarkDef(size=100))
    .encode(x = "gameweek:O", y = "total_points:Q",
            tooltip = [
                alt.Tooltip('name:N', title = 'Name'),
                alt.Tooltip('total_points:Q', title = 'Points Scored')
            ])
    .add_params(selection)
    .transform_filter
        (alt.datum['team_player'] == selection)
    .interactive()
)


points_by_gameweek

alt.Chart(...)

In [13]:

## Price by Gameweek


alt.data_transformers.enable('vegafusion')

price_by_gameweek = (
    alt.Chart(gameweek_data)
    .transform_calculate(value = 'datum.value/10') ## reflects pound value of player
    .mark_line(
        point=alt.OverlayMarkDef(size = 100)
    )
    .encode(x="gameweek:O", y="value:Q",
            tooltip=[
                alt.Tooltip('name:N', title = "Name"),
                alt.Tooltip('value:Q', title = "Price")
            ])
    .add_params(selection)
    .transform_filter(alt.datum['team_player'] == selection)
)

price_by_gameweek




alt.Chart(...)

In [14]:
## Points/price visualization

alt.data_transformers.enable('vegafusion')

points_per_pound = (
    alt.Chart(gameweek_data)
    .transform_calculate(
        value = 'datum.value/10',
    )
    .transform_calculate(
        points_per_pound = 'datum.total_points/datum.value'
    )
    .mark_line(
        point = alt.OverlayMarkDef(size = 100)
    )
    .encode(
        x = "gameweek:O", y = "points_per_pound:Q",
        tooltip= [
            alt.Tooltip('name:N', title = "Name"),
            alt.Tooltip('points_per_pound:Q', title = "Price per Pound", format = '.2f')
        ]
    )
    .add_params(selection)
    .transform_filter(alt.datum['team_player'] == selection)
)


points_per_pound


alt.Chart(...)

In [15]:
## XP v Points Scored Chart

alt.data_transformers.enable('vegafusion')

line_scale = alt.Scale(domain=['Actual', 'Expected'], range=['#1f77b4', '#7f7f7f'])

tooltip = [
    alt.Tooltip('name:N', title='Name'),
    alt.Tooltip('gameweek:O', title='Gameweek'),
    alt.Tooltip('total_points:Q', title='Actual'),
    alt.Tooltip('xP:Q', title='Expected', format='.1f'),
    alt.Tooltip('diff:Q', title='Difference', format='+.1f')
]

xp_vs_actual = (
    alt.layer(
        # Green shade: actual above expected
        alt.Chart().mark_area(opacity=0.3, color='#2ca02c').encode(
            x='gameweek:O', y='xP:Q', y2='upper:Q'
        ),
        # Red shade: actual below expected
        alt.Chart().mark_area(opacity=0.3, color='#d62728').encode(
            x='gameweek:O', y='lower:Q', y2='xP:Q'
        ),
        # Expected line (dashed)
        alt.Chart()
        .transform_calculate(metric="'Expected'")
        .mark_line(point=alt.OverlayMarkDef(size=100), strokeDash=[5, 4])
        .encode(
            x='gameweek:O',
            y='xP:Q',
            color=alt.Color('metric:N', scale=line_scale, title=None),
            tooltip=tooltip
        ),
        # Actual line (solid)
        alt.Chart()
        .transform_calculate(metric="'Actual'")
        .mark_line(point=alt.OverlayMarkDef(size=100))
        .encode(
            x=alt.X('gameweek:O', title='Gameweek'),
            y=alt.Y('total_points:Q', title='Points'),
            color=alt.Color('metric:N', scale=line_scale, title=None),
            tooltip=tooltip
        ),
        data=gameweek_data
    )
    .transform_filter(alt.datum['team_player'] == selection)
    .transform_aggregate(
        total_points='sum(total_points)',
        xP='sum(xP)',
        groupby=['gameweek', 'name']
    )
    .transform_calculate(
        upper='max(datum.total_points, datum.xP)',
        lower='min(datum.total_points, datum.xP)',
        diff='datum.total_points - datum.xP'
    )
    .add_params(selection)
)

xp_vs_actual


alt.LayerChart(...)

In [17]:
## Threat Line Chart

alt.data_transformers.enable('vegafusion')

threat_by_week = (
    alt.Chart(gameweek_data)
    .add_params(selection)
    .mark_line(point=alt.OverlayMarkDef(size=100))
    .encode(
        x = 'gameweek:O', y = 'threat:Q',
        tooltip = [
            alt.Tooltip('name:N', title = 'Name'),
            alt.Tooltip('threat:Q', title="Threat"),
            alt.Tooltip('total_points:Q', title="Points Scored")
        ]
    )
    .transform_filter(alt.datum['team_player'] == selection)

)

threat_by_week


alt.Chart(...)

In [18]:
## Histogram - bin highlights the player

alt.data_transformers.enable('vegafusion')

player_options=sorted(player_data['name'].unique())

player_select = alt.param(
    name = 'player_select',
    value = player_options[0],
    bind=alt.binding_select(options=player_options, name='Player: ')
)

points_histogram = (
    alt.Chart(player_data)
    .transform_filter('datum.minutes > 0')
    .transform_calculate(is_player='datum.name == player_select ? 1 : 0')
    .transform_joinaggregate(pos_had_player='max(is_player)', groupby=['element_type'])
    .transform_filter('datum.pos_had_player == 1')
    .transform_bin('points_bin', 'total_points', bin=alt.Bin(step=10))
    .transform_joinaggregate(bin_has_player='max(is_player)', groupby=['points_bin'])
    .transform_aggregate(
        players='count()',
        groupby=['points_bin', 'points_bin_end', 'bin_has_player', 'element_type']
    )
    .transform_calculate(
        group="datum.bin_has_player == 1 ? 'Selected player' : 'Other players'"
    )
    .mark_bar(stroke='white')
    .encode(
        x=alt.X('points_bin:Q', bin = 'binned', title='Total points'),
        x2='points_bin_end:Q',
        y=alt.Y('players:Q', title = 'Number of Players'),
        color=alt.Color(
            'group:N',
            scale=alt.Scale(domain=['Selected player', 'Other players'],
                            range=['#e45756', '#c7c7c7']),
            title=None
        ),
        tooltip=[
            alt.Tooltip('element_type:N', title = 'Position'),
            alt.Tooltip('points_bin:Q', title='From'),
            alt.Tooltip('points_bin_end:Q', title='To'),
            alt.Tooltip('players:Q', title='Players')
        ]
    )
    .add_params(player_select)
)

points_histogram



alt.Chart(...)

In [19]:
## Stacked Bar Chart for Each Player

def points_breakdown(df):
    pos = df['position']
    mins = df['minutes']
    out = pd.DataFrame(index=df.index)

    out['Appearance'] = np.select([mins >= 60, mins > 0], [2, 1], 0)
    out['Goals'] = df['goals_scored'] * pos.map({'GK': 10, 'DEF': 6, 'MID': 5, 'FWD': 4})
    out['Assists'] = df['assists'] * 3
    out['Clean sheets'] = df['clean_sheets'] * pos.map({'GK': 4, 'DEF': 4, 'MID': 1, 'FWD': 0})
    out['Saves'] = (df['saves'] // 3) + df['penalties_saved'] * 5
    dc_threshold = pos.map({'GK': np.inf, 'DEF': 10, 'MID': 12, 'FWD': 12})
    out['Defensive contribution'] = np.where(df['defensive_contribution'] >= dc_threshold, 2, 0)
    out['Bonus'] = df['bonus']
    out['Goals conceded'] = np.where(pos.isin(['GK', 'DEF']), -(df['goals_conceded'] // 2), 0)
    out['Cards'] = -df['yellow_cards'] - 3 * df['red_cards']
    out['Other'] = -2 * df['own_goals'] - 2 * df['penalties_missed']

    # Anything the rules above miss goes into Other, so each bar always equals total_points
    out['Other'] += df['total_points'] - out.sum(axis=1)
    return out

categories = ['Appearance', 'Goals', 'Assists', 'Clean sheets', 'Saves',
              'Defensive contribution', 'Bonus', 'Goals conceded', 'Cards', 'Other']

id_cols = ['team_player', 'name', 'position', 'gameweek']
breakdown_long = (
    pd.concat([gameweek_data[id_cols], points_breakdown(gameweek_data)], axis=1)
    .melt(id_vars=id_cols, value_vars=categories, var_name='category', value_name='points')
)

points_by_category = (
    alt.layer(
        alt.Chart()
        .transform_aggregate(points='sum(points)', groupby=['name', 'category'])
        .transform_calculate(
            cat_order=f"indexof({categories}, datum.category)",
            earned='max(datum.points, 0)',           # positive points only
            abs_points='abs(datum.points)'
        )
        .transform_joinaggregate(
            net_total='sum(points)',
            earned_total='sum(earned)',
            abs_total='sum(abs_points)',
            groupby=['name']
        )
        .transform_calculate(
            pct_of_net='datum.net_total != 0 ? datum.points / datum.net_total : null',
            pct_of_earned='datum.earned_total > 0 ? datum.points / datum.earned_total : null',
            pct_of_abs='datum.abs_total > 0 ? datum.points / datum.abs_total : null'
        )
        .mark_bar(height=40)
        .encode(
            x=alt.X('points:Q', title='Total points'),
            y=alt.Y('name:N', title=None),
            color=alt.Color('category:N',
                            scale=alt.Scale(domain=categories, scheme='tableau10'),
                            title='Category'),
            order=alt.Order('cat_order:Q'),
            tooltip=[
                alt.Tooltip('name:N', title='Name'),
                alt.Tooltip('category:N', title='Category'),
                alt.Tooltip('points:Q', title='Points'),
                alt.Tooltip('pct_of_earned:Q', title='% of points earned', format='.1%'),
                alt.Tooltip('pct_of_net:Q', title='% of net total', format='.1%')
            ]
        ),
        alt.Chart().mark_rule(color='black').encode(x=alt.datum(0)),   # zero line
        data=breakdown_long
    )
    .transform_filter(alt.datum['team_player'] == selection)
    .properties(width=450, height=80)
    .add_params(selection)
)

points_by_category



alt.LayerChart(...)

In [20]:
## Donut Chart - proportion of points scored (goals, clean sheets, defcon, etc.)


donut_stats = ['total_points', 'minutes', 'value_per_m', 'ict_index', 'bps', 'threat', 'creativity']
stat_labels = {
    'total_points': 'Total points', 'minutes': 'Minutes played',
    'value_per_m': 'Points per £m', 'ict_index': 'ICT index',
    'bps': 'BPS', 'threat': 'Threat', 'creativity': 'Creativity'
}

arcs = alt.Chart().mark_arc(innerRadius=45, outerRadius=65).encode(
    theta=alt.Theta('share:Q', stack=True),
    color=alt.Color('fill:N', scale=None),          # use the colors calculated below as-is
    order=alt.Order('part_order:Q'),
    tooltip=[
        alt.Tooltip('name:N', title='Name'),
        alt.Tooltip('stat_label:N', title='Stat'),
        alt.Tooltip('value:Q', title='Value', format=',.1f'),
        alt.Tooltip('pct:Q', title='Percentile', format='.0%'),
        alt.Tooltip('element_type:N', title='Compared with')
    ]
)

label = (
    alt.Chart()
    .transform_filter("datum.part == 'filled'")     # one label per donut
    .mark_text(fontSize=20, fontWeight='bold')
    .encode(text='pct_label:N')
)

percentile_donuts = (
    alt.layer(arcs, label)
    .properties(width=150, height=150)
    .facet(
        facet=alt.Facet('stat_label:N', title=None,
                        sort=[stat_labels[s] for s in donut_stats],
                        header=alt.Header(labelFontSize=13, labelFontWeight='bold')),
        columns=3,
        data=player_data
    )
    .transform_filter(f'datum.minutes > {MIN_MINUTES}')
    .transform_fold(donut_stats, as_=['stat', 'value'])
    .transform_window(
        pct='percent_rank()',
        sort=[alt.SortField('value', order='ascending')],
        groupby=['element_type', 'stat']
    )
    .transform_filter('datum.name == player_select')
    .transform_calculate(
        stat_label=f"{stat_labels}[datum.stat]",
        n='round(datum.pct * 100)',
        pct_label="datum.n + ((datum.n % 100 >= 11 && datum.n % 100 <= 13) ? 'th' : "
                  "datum.n % 10 == 1 ? 'st' : datum.n % 10 == 2 ? 'nd' : "
                  "datum.n % 10 == 3 ? 'rd' : 'th')",
        filled='datum.pct',
        remaining='1 - datum.pct'
    )
    .transform_fold(['filled', 'remaining'], as_=['part', 'share'])
    .transform_calculate(
        part_order="datum.part == 'filled' ? 0 : 1",
        fill="datum.part == 'remaining' ? '#e6e6e6' : "
             "datum.pct >= 0.67 ? '#2ca02c' : datum.pct >= 0.33 ? '#f2b701' : '#d62728'"
    )
    .add_params(player_select)
)

percentile_donuts


alt.FacetChart(...)

In [22]:
## Leaderboards

import json
leaderboard_metrics = {
    'total_points': 'Total points',
    'value_per_m': 'Points per £m',
    'goals_scored': 'Goals',
    'assists': 'Assists',
    'clean_sheets': 'Clean sheets',
    'bonus': 'Bonus points',
    'bps': 'BPS',
    'ict_index': 'ICT index',
    'threat': 'Threat',
    'creativity': 'Creativity',
    'influence': 'Influence',
    'minutes': 'Minutes played',
    'selected_by_percent': 'Ownership %'
}
TOP_N = 10

metric_select = alt.param(
    name='metric_select',
    value='total_points',
    bind=alt.binding_select(options=list(leaderboard_metrics),
                            labels=list(leaderboard_metrics.values()),
                            name='Metric: ')
)
position_select = alt.param(
    name='position_select',
    value='All',
    bind=alt.binding_select(options=['All', 'GK', 'DEF', 'MID', 'FWD'], name='Position: ')
)

row_y = alt.Y('rank:O', axis=None)
highlight = alt.condition('datum.name == player_select',
                          alt.value('#e45756'), alt.value('#333333'))

def column(field, header, width, align='left', fmt=None):
    """One text column of the table."""
    text = alt.Text(field, format=fmt) if fmt else alt.Text(field)
    return (
        alt.Chart()
        .mark_text(align=align, baseline='middle', fontSize=13,
                   x=0 if align == 'left' else width)
        .encode(y=row_y, text=text, color=highlight)
        .properties(
            width=width,
            height=TOP_N * 28,
            title=alt.Title(header, anchor='start' if align == 'left' else 'end',
                            fontSize=12, color='#666')
        )
    )

leaderboard_table = (
    alt.hconcat(
        column('rank:O', '#', 30),
        column('name:N', 'Player', 230),
        column('element_type:N', 'Pos', 50),
        column('price_label:N', 'Price', 70, align='right'),
        column('total_points:Q', 'Points', 60, align='right'),
        column('value:Q', 'Value', 80, align='right', fmt=',.1~f'),
        data=player_data,
        spacing=0
    )
    .transform_filter(f'datum.minutes > {MIN_MINUTES}')
    .transform_filter("position_select == 'All' || datum.element_type == position_select")
    .transform_fold(list(leaderboard_metrics), as_=['metric', 'value'])
    .transform_filter('datum.metric == metric_select')
    .transform_window(rank='row_number()', sort=[alt.SortField('value', order='descending')])
    .transform_filter(f'datum.rank <= {TOP_N} || datum.name == player_select')
    .transform_calculate(price_label="'£' + format(datum.now_cost / 10, '.1f') + 'm'")
    .properties(
        title=alt.Title(alt.expr(f"{json.dumps(leaderboard_metrics)}[metric_select] + ' leaderboard'"),
                        anchor='start', fontSize=16)
    )
    .add_params(metric_select, position_select, player_select)
    .configure_view(stroke=None)      # removes the box around each column
)

leaderboard_table


alt.HConcatChart(...)

In [30]:
## Next Opponents - strength of team???

# One row per team per fixture (the joined data has a row for every player)
fixtures = gameweek_data[['team', 'gameweek', 'fixture', 'opponent', 'opponent_strength']].drop_duplicates()

team_options = sorted(fixtures['team'].unique())
team_select = alt.param(
    name='team_select',
    value=team_options[0],
    bind=alt.binding_select(options=team_options, name='Team: ')
)

opponent_strength_chart = (
    alt.layer(
        # Grey line connecting the gameweeks
        alt.Chart().mark_line(color='#bbbbbb'),
        # Colored points: 4-5 red, 2-3 green
        alt.Chart().mark_point(filled=True, size=120, opacity=1).encode(
            color=alt.condition('datum.opponent_strength >= 4',
                                alt.value('#d62728'),   # red
                                alt.value('#2ca02c')),  # green
            tooltip=[
                alt.Tooltip('gameweek:O', title='Gameweek'),
                alt.Tooltip('opponent:N', title='Opponent'),
                alt.Tooltip('opponent_strength:Q', title='Opponent strength')
            ]
        ),
        data=fixtures
    )
    .encode(
        x=alt.X('gameweek:O', title='Gameweek', axis=alt.Axis(labelAngle=0)),
        y=alt.Y('opponent_strength:Q', title='Opponent strength',
                scale=alt.Scale(domain=[1, 5]),
                axis=alt.Axis(values=[1, 2, 3, 4, 5], format='d'))
    )
    .transform_filter(alt.datum.team == team_select)
    .properties(width=500, height=250)
    .add_params(team_select)
)

opponent_strength_chart


alt.LayerChart(...)